# Video generation: input → output

One dog walking, a simple background and one steady shot. Use a GPU runtime on Colab. The existing GIF samples below are completed four-step runs. A new dog output and eight-step low-VRAM measurements are pending GPU availability.


## Setup

Colab uses the project folder below. For local Jupyter, change `PROJECT` to your repository directory. Generation runs in the pinned Python 3.12 environment through `uv`.


In [ ]:
from pathlib import Path
import subprocess
import sys
import shutil

PROJECT = Path("/content/7-8gb-vram-video-generation")
# Local Jupyter: set PROJECT to the absolute path of your cloned repository.

if not (PROJECT / "main.py").exists():
    subprocess.run(["git", "clone", "https://github.com/ujjwal-basnet/7-8gb-vram-video-generation.git", str(PROJECT)], check=True)
if shutil.which("uv") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
subprocess.run(["uv", "sync", "--locked"], cwd=PROJECT, check=True)


def run_script(script, *args):
    subprocess.run(["uv", "run", "python", script, *map(str, args)], cwd=PROJECT, check=True)


## Input

Choose four or eight Turbo steps. Four-step is measured; eight-step is available for testing. 39 frames is about 1.63 seconds; 124 frames is about 5.17 seconds. Output is 640×384 at 24 fps with audio. A continuous 15-second render under 8 GB remains unverified. Edit the prompt directly below.


In [ ]:
STEPS = 4  # @param [4, 8]
FRAMES = 124  # @param [22, 39, 56, 73, 90, 107, 124]
SEED = 20261008  # @param {type:"integer"}

PROMPT = 'One adult golden retriever with light golden fur, dark brown eyes, a black nose and floppy ears. The dog walks slowly from left to right along a flat dirt path beside plain green grass. Realistic fur and natural four-legged walking gait. Keep the whole dog visible, with consistent size, coat and anatomy. Only one dog. No people, other animals, props, buildings, text or logos. Soft daylight. One continuous take with no cuts. Quiet paw steps and a gentle breeze, no dialogue or music. Full-body side view. A steady camera tracks gently alongside the dog at its walking pace.'

print(PROMPT)
print(f"Turbo steps: {STEPS}; frames: {FRAMES}; seed: {SEED}")


## Download models

This checks GPU access before downloading the selected adapter and pinned model components.


In [ ]:
subprocess.run(["nvidia-smi"], check=True)
run_script("download_models.py", "--steps", STEPS)


## Generate one video

One generation produces one clip. The notebook does not join separate videos.


In [ ]:
OUTPUT = PROJECT / "output/dog-walking.mp4"
OUTPUT.parent.mkdir(exist_ok=True)
INPUT = OUTPUT.parent / "input-prompt.txt"
INPUT.write_text(PROMPT)
run_script("monitor_run.py", "--prompt-file", INPUT,
           "--steps", STEPS, "--frames", FRAMES, "--seed", SEED, "--output", OUTPUT)


## Your output

Run this cell after generation succeeds to play the new video with audio.


In [ ]:
from IPython.display import Video, display
display(Video(str(OUTPUT), embed=True))


## Existing sample: input and output

The completed short reference used four Turbo steps, 39 frames, seed 9175 and SelfLift from 512×320 to 640×384. Its prompt was:

> An adult female storm guardian with warm olive skin, sharp amber eyes and long jet-black hair in a single high braid. She wears dark teal lamellar armor with engraved bronze-gold edges and an indigo silk cape. She carries a bronze spear with a glowing cyan crystal tip. High-end 3D CG cinematic fantasy with a Xianxia-inspired art direction, physically based bronze armor and woven silk, realistic skin shading, cool volumetric cloud light and bright cyan lightning. No dialogue, subtitles, logos or music. A five-second establishing sequence at a ruined floating temple above an endless sea of storm clouds. Start with a low-angle medium full-body view of the guardian standing on cracked black basalt tiles, hair and cape moving naturally in the wind. The camera slowly tracks forward as she raises her spear. Thin cyan electric arcs gather around its crystal tip. At the far end of the courtyard, two enormous carved stone sentinels awaken, their eyes glowing blue. Maintain clear anatomy, readable silhouette and one continuous deliberate motion. Natural wind, distant thunder and a subtle electric hum.

Output preview (GIF has reduced resolution/frame rate and no audio):

![Existing short output](https://raw.githubusercontent.com/ujjwal-basnet/7-8gb-vram-video-generation/main/samples/short-scene.gif)


## Existing 15-second sample

This historical sample is preserved as requested. It is an assembled reference film from the earlier workflow, not a new dog output or an eight-step benchmark.

![Existing 15-second sample](https://raw.githubusercontent.com/ujjwal-basnet/7-8gb-vram-video-generation/main/samples/storm-guardian.gif)
